In [2]:
# import torch dataset and dataloader
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
# Import from_smiles from pytorch geometric
from torch_geometric.utils import from_smiles
from all_functions import get_node_labels
from rdkit import Chem

# def get_node_labels(protac_smiles, substructure_smiles):                         #This did not work for me. error invalid type '' in dataloader. I wrote another version of get_node_labels in all_functions.
#     idx = boundary_ligand_nodes(protac_smiles, substructure_smiles)
#     boundary_POI_node_index, boundary_E3_node_index = idx
#     num_atoms = Chem.MolFromSmiles(protac_smiles).GetNumAtoms()
#     node_labels = np.zeros((num_atoms, 1))
#     POI_LABEL = 1
#     E3_LABEL = 2
#     node_labels[boundary_POI_node_index] = POI_LABEL
#     node_labels[boundary_E3_node_index] = E3_LABEL
#     return node_labels
    

class ProtacDataset(Dataset):

    def __init__(self, protac_df, transform=None):
        self.protac_df = protac_df
        self.protac_smiles = protac_df['PROTAC SMILES'].tolist()
        self.poi_smiles = protac_df['POI SMILES'].tolist()
        self.e3_smiles = protac_df['E3 SMILES'].tolist()
        self.substructures = ['.'.join(protac_df[['POI SMILES', 'LINKER SMILES','E3 SMILES']].iloc[i].tolist()) for i in range(len(protac_df))] #remove later when I have updated functions to not use split_sort. Possible as the substructures will be already separated in the dataframe

        self.node_boundaries = [get_node_labels(
            smiles, substructures_joined) for smiles, substructures_joined in zip(self.protac_smiles, self.substructures)]
    
    def __len__(self):
        return len(self.protac_df)
    
    def __getitem__(self, idx):
        elem = {
            'pyg_data': from_smiles(self.protac_smiles[idx]), # (1024,)
            'node_boundaries': self.node_boundaries[idx], # List of boundary classes for each node # (num_nodes,) # boundary_ligand_nodes
        }
        return elem


In [3]:
protac_pub_trainset_df = pd.read_csv('../../data/augmented/protac_pub_testset_testing.csv')
protac_pub_testset_df = pd.read_csv('../../data/augmented/protac_pub_trainset_testing.csv')

In [4]:
train_set_pub = ProtacDataset(protac_df=protac_pub_trainset_df)

In [ ]:

def PROTACSplitter(nn.Module):

    def __init__(self, ):

    
    def forward(self, batch):
        data = batch['pyg_data']
        x = self.gnn_layer(data.x, data.edge_index, data.edge_attr)
        x = self.linear_layer(x)
        return x


In [5]:
from torch_geometric.loader import DataLoader
#train_loader = DataLoader(train_set_pub, batch_size=32, shuffle=True)
#next(iter(train_loader))

{'pyg_data': DataBatch(x=[1539, 9], edge_index=[2, 3366], edge_attr=[3366, 3], smiles=[27], batch=[1539], ptr=[28]),
 'node_boundaries': [tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0]),
  tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0]),
  tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0]),
  tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0]),
  tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0]),
  tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0]),
  tensor([0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0,
          0, 1, 0]),
  tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0]),
  tensor([0, 0, 2, 0, 0, 0, 1, 2, 2, 2, 

In [ ]:


model = PROTACSplitter()
for batch in DataLoader(train_set_pub, batch_size=32):
    # batch['protac_smiles'] = (batch_size, 1024)
    y_hat = model(batch)
    loss = loss_fn(y_hat, batch['node_boundaries'])
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Metric
    y_hat = torch.argmax(y_hat, dim=1) # (batch_size, num_nodes) -> (batch_size, 1)
    acc = (y_hat == batch['node_boundaries']).sum() / len(y_hat)
